<img src="https://cdn.jsdelivr.net/gh/maxischa/datacamp_test@5a33b79/ressources/img/logo_macmia.png" alt="Banque des Territoires · France 2030 · MACMIA" width="520">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc2_donnees_v2/corrections/seance1_correction.ipynb)

# Séance 2.1 — Charger, comprendre et nettoyer une base de données

**Correction** · durée : 6h (2h de cours, 4h d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- charger un fichier de données depuis le web en une ligne
- décrire un fichier que vous n'avez jamais vu en 30 secondes
- sélectionner exactement les lignes et les colonnes qui vous intéressent
- calculer des indicateurs simples sur une colonne entière
- lire un message d'erreur au lieu de le subir

## Le contexte

Vous venez d'arriver chez un **détaillant en ligne** européen. On vous remet
l'historique des ventes de l'année écoulée et une question simple :

> *« Sur quel marché faut-il investir l'an prochain ? »*

Vous ne pouvez pas répondre tant que vous ne savez pas ce que contient ce
fichier. Cette séance, c'est exactement ça : **prendre en main un jeu de
données qu'on n'a jamais vu**.

Nous avons trois fichiers :

| Fichier | Une ligne = | Colonnes |
|---|---|---|
| `ventes.csv` | un produit dans une commande | `date`, `cmd_id`, `prod_id`, `qte`, `prix`, `client_id` |
| `clients.csv` | un client | `client_id`, `pays`, `segment`, `date_insc` |
| `produits.csv` | un produit | `prod_id`, `libelle`, `categorie` |

> 📋 **Comment on travaille.** Pour chaque technique : une démonstration que
> vous suivez, puis **deux exercices que vous faites** — le premier a des
> `____` à remplir, le second est une cellule vide où vous écrivez tout. La
> cellule de vérification vous dit tout de suite si votre réponse est bonne.

## 1. Le point de départ

Cette cellule est présente au début de **tous** les notebooks du cours. Elle
charge les outils dont nous aurons besoin et règle l'affichage pour les petits
écrans. Exécutez-la (bouton ▶) sans chercher à la comprendre pour l'instant.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://cdn.jsdelivr.net/gh/maxischa/datacamp_test@5a33b79/bloc2_donnees/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Les trois premières lignes chargent des **bibliothèques** : du code déjà écrit
par d'autres, que vous appelez ensuite en une instruction. Le `as pd` leur
donne un **surnom** — tout le monde utilise les mêmes, et tous les exemples que
vous trouverez en ligne les emploient.

| La ligne | Ce qu'elle apporte | Quand elle sert |
|---|---|---|
| `import pandas as pd` | les **tableaux** : charger, filtrer, regrouper, calculer | dès aujourd'hui, et partout ensuite |
| `import numpy as np` | le **calcul sur des colonnes entières** : classer, remplacer, marquer des valeurs manquantes | en 2.2, pour nettoyer |
| `import matplotlib.pyplot as plt` | les **graphiques** : courbes, barres, histogrammes | en 2.3, pour visualiser |

**`pandas` est celle qui compte.** Pensez-y comme à **un Excel qu'on pilote par
des instructions** : même objet — un tableau de lignes et de colonnes — mais au
lieu de cliquer, on écrit ce qu'on veut. L'avantage : ça marche sur 45 000
lignes aussi vite que sur 10, et on peut relancer exactement la même analyse le
mois prochain.

Les deux autres attendront leur tour. Elles sont dans la cellule dès
aujourd'hui parce que **cette cellule est la même dans tous les notebooks du
cours** : vous n'aurez jamais à vous demander laquelle exécuter.

Les deux `set_option` qui suivent règlent la **largeur d'affichage** : sans
elles, un tableau de six colonnes part en accordéon sur une tablette. La
dernière ligne, `BASE`, est l'adresse web du dossier de données — c'est elle
qui vous évite de télécharger quoi que ce soit.

## 2. Charger les données

Une seule ligne. `pd.read_csv()` accepte directement une **adresse web** :
rien à télécharger, rien à ranger dans un dossier.

In [ ]:
ventes = pd.read_csv(BASE + "ventes.csv")  ## lit le fichier en ligne
ventes.head(3)                             ## les 3 premieres lignes

`ventes` est un **DataFrame** : le nom que pandas donne à un tableau.

`.head(3)` affiche les 3 premières lignes. Toujours commencer par là : c'est
la façon la plus rapide de vérifier que le fichier est bien celui qu'on croit.

> 💡 `head()` prend en argument le **nombre de lignes** à afficher :
> `head(3)` en montre trois, `head(20)` en montre vingt. Sans argument, elle
> en affiche cinq.

## 3. La carte d'identité d'un fichier

Deux commandes, tout de suite après le chargement. En 30 secondes vous savez
à quoi vous avez affaire.

In [ ]:
ventes.shape   ## (nombre de lignes, nombre de colonnes)

In [ ]:
ventes.info()  ## les colonnes, leur type, les valeurs manquantes

Ce que `info()` vous dit, ligne par ligne :

- **`45123 entries`** — 45 123 lignes.
- **`non-null`** — combien de valeurs sont renseignées. Ici tout est complet ;
  on verra en séance 2.2 que c'est très rare dans la vraie vie.
- **`Dtype`** — le *type* de chaque colonne, et c'est le plus important :
  - `int64` : nombre entier (`qte`, `client_id`)
  - `float64` : nombre à virgule (`prix`)
  - `object` : **du texte** (`date`, `prod_id`)

> ⚠️ Regardez `date` : son type est `object`, c'est-à-dire du **texte**. Pour
> pandas, `"2011-10-04"` est une chaîne de caractères, pas une date. On ne peut
> donc pas encore lui demander « quel mois ? ». On corrigera ça en séance 2.2.

In [ ]:
# 45 123 lignes, mais combien de clients et de commandes distincts ?
print("clients   :", ventes["client_id"].nunique())   ## valeurs distinctes
print("commandes :", ventes["cmd_id"].nunique())      ## idem sur la commande

**Une ligne n'est pas un client.** Une ligne est *un produit dans une
commande*. 45 123 lignes correspondent à 1 955 commandes passées par 472
clients.

C'est la première question à se poser devant n'importe quel fichier :
**une ligne, c'est quoi exactement ?** Se tromper là-dessus, c'est se tromper
sur tout le reste de l'analyse.

---

### ✏️ Exercice — Le catalogue produits

> **Votre mission :**
> - Charger `produits.csv` → `produits`, puis afficher ses 3 premières lignes.
> - Combien de références contient le catalogue ? → `nb_produits`
> - *Rappel :* `df.shape` renvoie `(lignes, colonnes)` ; `shape[0]` est le nombre de lignes.

In [ ]:
produits = pd.read_csv(BASE + "produits.csv")   ## meme commande, autre fichier
nb_produits = produits.shape[0]   ## [0] = les lignes, [1] = les colonnes

print(nb_produits, "references au catalogue")
produits.head(3)

In [ ]:
verifier("references au catalogue", nb_produits == 2956,
         "shape[0] donne les lignes, shape[1] les colonnes")

---

### ✏️ Exercice — Les deux bouts du fichier

> **Votre mission :**
> - Afficher les **25 premières** lignes de `ventes`, puis les **10 dernières**.
> - Mettre la valeur de `date` de la toute dernière ligne dans `derniere_date`.
> - Le fichier couvre-t-il des périodes comparables d'un bout à l'autre ? Regardez le mois de la dernière ligne.
> - *Nouveau :* `df.tail(n)` fait ce que `head(n)` fait, mais par la fin.

In [ ]:
print(ventes.head(25))   ## le debut : ce qu'on regarde toujours

fin = ventes.tail(10)   ## la fin : ce qu'on oublie presque toujours
derniere_date = fin["date"].iloc[-1]   ## la toute derniere ligne
print(derniere_date)
fin

In [ ]:
verifier("derniere date du fichier", derniere_date.startswith("2011-12-09"),
         "tail(10) puis la colonne date de la derniere ligne")

Un fichier est presque toujours trié par date. Regarder la fin, c'est vérifier
qu'il ne s'arrête pas **au milieu d'une période** : ici il s'arrête le
9 décembre. Décembre est donc incomplet dans ce fichier — retenez-le, ça
comptera en séance 2.3, quand on le tracera.

## 4. Choisir ce qu'on regarde

### Une colonne

In [ ]:
ventes["prix"].head(3)   ## une seule colonne : c'est une Series

### Plusieurs colonnes

Doubles crochets : les crochets extérieurs veulent dire « je sélectionne »,
les intérieurs délimitent la **liste** des colonnes voulues.

In [ ]:
ventes[["prod_id", "qte", "prix"]].head(3)  ## une liste -> un tableau

### Naviguer grâce aux lignes du Dataset— `.loc`

`.loc` raisonne en **étiquettes** : le nom de la ligne, le nom de la colonne.
Ici les lignes n'ont pas reçu de nom, alors pandas leur a donné leur numéro
d'arrivée. C'est pour ça que `.loc[0]` et `.iloc[0]` renvoient la même ligne —
mais **c'est une coïncidence**, pas une règle, et elle ne survit pas au
premier filtrage.

In [ ]:
ventes.loc[10, "prix"]   ## ligne d'etiquette 10, colonne nommee "prix"

In [ ]:
# Avec .loc on nomme les colonnes au lieu de les compter
ventes.loc[0:2, ["qte", "prix"]]   ## et la borne 2 est INCLUSE

---

### ✏️ Exercice — Sélectionner avec `.loc`

> **Votre mission :**
> - Afficher toutes les lignes des colonnes **`cmd_id`**, **`prod_id`** et **`qte`** de `ventes` → `colonnes`.
> - Afficher les lignes dont la quantité **`qte` est supérieure à 2**, en conservant uniquement les colonnes **`prod_id`**, **`qte`** et **`prix`** → `ventes_filtrees`.
> - Mettre le nombre de lignes de `ventes_filtrees` dans `n_ventes`.
> - *Nouveau :* avec `.loc`, on écrit `df.loc[lignes, colonnes]` — les lignes avant la virgule, les colonnes après. Un `:` seul signifie « toutes les lignes ».

In [ ]:
colonnes = ventes.loc[:, ["cmd_id", "prod_id", "qte"]]      ## : = toutes les lignes
ventes_filtrees = ventes.loc[ventes["qte"] > 2,
                             ["prod_id", "qte", "prix"]]    ## condition, puis colonnes
n_ventes = len(ventes_filtrees)

print(colonnes.shape, "->", n_ventes, "lignes de plus de 2 unites")
ventes_filtrees.head(3)

In [ ]:
verifier(
    "selection des trois colonnes",
    list(colonnes.columns) == ["cmd_id", "prod_id", "qte"]
    and len(colonnes) == len(ventes),
    '.loc[:, ["cmd_id", "prod_id", "qte"]] : avant la virgule les lignes, apres les colonnes'
)

verifier(
    "filtrage des ventes",
    ventes_filtrees.equals(
        ventes.loc[
            ventes["qte"] > 2,
            ["prod_id", "qte", "prix"]
        ]
    ),
    'placez la condition ventes["qte"] > 2 avant la virgule et les colonnes apres'
)

verifier(
    "nombre de ventes filtrees",
    n_ventes == len(ventes_filtrees),
    "utilisez len(ventes_filtrees)"
)

## 5. Ne garder qu'une partie des lignes — `.query()`

On veut les ventes dont le prix dépasse 50 € :

In [ ]:
cheres = ventes.query("prix > 50")   ## la condition, entre guillemets
print(cheres.shape)                  ## combien de lignes ont survecu ?
cheres.head(3)

`.query()` prend une **condition écrite entre guillemets**, presque en français :
`"prix > 50"`, `"qte >= 100"`, `"pays == 'France'"`.

Vous rencontrerez aussi l'autre écriture, plus classique :

```python
ventes[ventes["prix"] > 50]
```

Les deux font exactement la même chose. **Nous utiliserons `.query()` dans ce
cours** : deux fois moins de ponctuation à taper, et beaucoup plus lisible dès
que la condition se complique.

### Une liste de valeurs — `in`

Pour retenir plusieurs valeurs d'une même colonne, inutile d'empiler les
`or` : `in` teste l'appartenance à une liste. Chargeons le fichier des
clients pour l'essayer sur des noms de pays.

In [ ]:
clients = pd.read_csv(BASE + "clients.csv")   ## une ligne = un client

# Attention aux guillemets : doubles a l'exterieur, simples a l'interieur
sud = clients.query("pays in ['Espagne', 'Portugal', 'Italie']")   ## in
print(len(sud), "clients dans ces trois pays")

> ⚠️ **Les guillemets imbriqués.** Toute la condition est une chaîne de
> caractères : guillemets **doubles à l'extérieur**, **simples à l'intérieur**
> pour les noms de pays. C'est la seule chose qui bloque vraiment sur cette
> tournure.

### Un intervalle

Un encadrement s'écrit comme en mathématiques, d'un seul tenant.

In [ ]:
moyennes = ventes.query("50 <= qte <= 100")   ## un encadrement
print(len(moyennes), "lignes")

### Ce qu'il y a dans les crochets — une colonne de Vrai/Faux

Prenez `ventes["prix"] > 50` tout seul, sans les crochets autour. Comparer une
colonne à une valeur ne rend pas **un** résultat : pandas pose la question à
**chacune des 45 123 lignes** et rend **une réponse par ligne**, `True` ou
`False`.

In [ ]:
depasse = ventes["prix"] > 50   ## une question posee a chaque ligne

print(depasse.head(3))   ## trois reponses, et le type : bool
print(depasse.sum(), "lignes repondent True")   ## .sum() les compte

Trois lignes, trois `False`, et en bas `dtype: bool` : c'est bien une colonne
de Vrai/Faux, aussi longue que le tableau. Et **41** — exactement le nombre de
lignes que `.query("prix > 50")` a gardées au début de cette section. C'est le
même objet : une fois pour compter, une fois pour filtrer.

Deux opérations se branchent dessus, et vous les retrouverez jusqu'au bloc 4 :

| Vous écrivez | Vous obtenez |
|---|---|
| `(ventes["prix"] > 50).sum()` | le **nombre** de lignes qui répondent `True` |
| `(ventes["prix"] > 50).mean()` | leur **part**, entre 0 et 1 |

> 💡 `.sum()` compte les `True` parce qu'un `True` vaut 1 et un `False` 0.
> C'est la tournure la plus fréquente du cours : « combien de lignes remplissent
> cette condition ? » s'écrit `(condition).sum()`.

---

### ✏️ Exercice — Les grosses commandes

> **Votre mission :**
> - Ne garder que les lignes de **plus de 100 unités** → `grosses`.
> - Combien y en a-t-il ? → `nb_grosses`

In [ ]:
grosses = ventes.query("qte > 100")   ## la condition entre guillemets
nb_grosses = grosses.shape[0]         ## shape[0] = le nombre de lignes

print(nb_grosses, "lignes de plus de 100 unites")

In [ ]:
verifier("grosses commandes", nb_grosses == 560,
         "query() prend la condition entre guillemets : \"qte > 100\"")

---

### ✏️ Exercice — Beaucoup d'unités, petit prix

> **Votre mission :**
> - Le service achats cherche les lignes qui partent **en volume à bas prix** : entre 50 et 100 unités, à moins de 2 € l'unité.
> - Combien y en a-t-il ? → `nb_volume`
> - Tout est à écrire : un encadrement **et** une seconde condition, dans la même chaîne. *Nouveau :* on empile deux conditions avec `and`.

In [ ]:
# Un encadrement et une condition supplementaire dans la meme chaine :
# query lit "50 <= qte <= 100 and prix < 2" comme une phrase
volume = ventes.query("50 <= qte <= 100 and prix < 2")
nb_volume = len(volume)

print(nb_volume, "lignes en volume a bas prix")

In [ ]:
verifier("volume a bas prix", nb_volume == 826,
         "un encadrement 50 <= qte <= 100, puis and prix < 2, dans la meme chaine")

## 6. Résumer en un coup d'œil

### `describe()` — le résumé chiffré

In [ ]:
# On selectionne les colonnes AVANT : sinon la sortie deborde de l'ecran
ventes[["qte", "prix"]].describe().round(2)   ## huit statistiques d'un coup

À lire ainsi :

- `mean` : la moyenne. Prix moyen : **3,93 €**.
- `50%` : la **médiane**, la valeur qui coupe la population en deux. **1,95 €**.
- `max` : la valeur maximale. **4 161 €**.

> 📊 Moyenne 3,93 € mais médiane 1,95 € : la moyenne est **deux fois** la
> médiane. C'est la signature d'une poignée de valeurs très élevées qui tirent
> la moyenne vers le haut. Devant un écart pareil, la médiane décrit bien mieux
> « le produit typique ». Un réflexe à garder : **comparer moyenne et médiane
> avant de citer un chiffre en réunion.**

### `value_counts()` — compter les catégories

In [ ]:
clients["segment"].value_counts()   ## deja trie du plus frequent

### Un seul chiffre à la fois

In [ ]:
print("prix moyen :", ventes["prix"].mean().round(2))   ## .round() arrondit
print("quantite max :", ventes["qte"].max())            ## le maximum
print("lignes a plus de 50 euros :", cheres.shape[0])   ## shape[0] = lignes

---

### ✏️ Exercice — Le pays le plus représenté

> **Votre mission :**
> - Compter les clients par pays → `par_pays`.
> - Mettre le nom du pays le plus représenté dans `pays_top` et son nombre de clients dans `nb_top`.
> - *Indice :* `value_counts()` trie déjà du plus fréquent au moins fréquent.

In [ ]:
par_pays = clients["pays"].value_counts()   ## deja trie

# .index donne les etiquettes, .iloc donne les valeurs par position
pays_top = par_pays.index[0]   ## le pays en tete
nb_top = par_pays.iloc[0]      ## son effectif

print(pays_top, ":", nb_top, "clients")

In [ ]:
verifier("pays le plus represente", pays_top == "Royaume-Uni",
         "value_counts() est deja trie : le premier est le plus frequent")
verifier("nombre de clients", nb_top == 235,
         ".index[0] donne l'etiquette, .iloc[0] donne l'effectif")

---

### ✏️ Exercice — La répartition, en pourcentage

> **Votre mission :**
> - Quelle **part** des clients chaque pays représente-t-il ? En %, arrondi à 1 décimale.
> - Afficher les cinq premiers, et mettre la part du Royaume-Uni dans `part_uk`.
> - *Nouveau :* `value_counts(normalize=True)` renvoie des parts (entre 0 et 1) au lieu d'effectifs.

In [ ]:
part = (clients["pays"].value_counts(normalize=True) * 100).round(1)   ## en %

part_uk = part["Royaume-Uni"]
print(part.head(5))

# Le Royaume-Uni pese la moitie du fichier clients a lui seul : c'est le
# marche domestique. On verra en 2.3 que sa part du CHIFFRE D'AFFAIRES
# n'est pas la meme que sa part des clients.

In [ ]:
verifier("part du Royaume-Uni", part_uk == 49.8,
         "normalize=True donne une part entre 0 et 1 : multipliez par 100")

## 7. Calculer sur des colonnes entières

Le chiffre d'affaires d'une ligne, c'est la quantité multipliée par le prix.

In [ ]:
ventes["ca"] = ventes["qte"] * ventes["prix"]  ## 45 123 calculs
ventes[["qte", "prix", "ca"]].head(3)          ## toujours verifier apres

Regardez bien ce qui vient de se passer : **une seule instruction a fait
45 123 multiplications**. On écrit l'opération *une fois, sur la colonne*, et
pandas l'applique à chaque ligne. C'est ce qui rend l'analyse de 45 000 lignes
aussi simple que celle de 10.

### Les opérations sur les nombres

Toutes les opérations arithmétiques fonctionnent de cette façon :

| Vous écrivez | Ce que pandas fait |
|---|---|
| `df["a"] + df["b"]` | additionne les deux colonnes, ligne à ligne |
| `df["a"] - 10` | retire 10 à **chaque** ligne |
| `df["a"] * 1.2` | multiplie chaque ligne par 1,2 |
| `df["a"] / df["a"].sum()` | divise chaque ligne par le total |
| `df["a"] ** 2` | met chaque ligne au carré |
| `df["a"].round(2)` | arrondit chaque ligne à 2 décimales |

In [ ]:
ventes["ttc"] = (ventes["prix"] * 1.2).round(2)   ## TVA a 20 %

ventes[["prix", "ttc", "qte", "ca"]].head(3)

Deux formes différentes cohabitent ici, et il faut les distinguer :

- `ventes["prix"] * 1.2` applique **le même nombre** à toutes les lignes ;
- `ventes["qte"] * ventes["prix"]` fait travailler **deux colonnes ensemble**,
  ligne par ligne.

### Et si on fait la même chose sur du texte ?

Les mêmes signes existent, mais ils ne veulent pas dire la même chose. Vous
l'aviez déjà croisé au bloc 1 avec `2 * "3"`.

In [ ]:
pays = clients["pays"]   ## une colonne de texte

print((pays + " (Europe ?)").head(2).tolist())   ## le + COLLE deux textes
print((pays * 2).head(2).tolist())               ## l'etoile REPETE le texte

Aucune erreur, et pourtant rien de ce qu'on attendait d'un `+` ou d'un `*`.

Le vrai danger n'est pas là : il est dans une colonne de **nombres stockés en
texte**. Le calcul « marche », et le résultat est absurde.

In [ ]:
# Comme le ferait un export mal configure : des quantites en texte
qte_txt = ventes["qte"].head(5).astype(str)

print("somme des nombres :", ventes["qte"].head(5).sum())
print("somme des textes  :", qte_txt.sum())   ## colle bout a bout

`96` d'un côté, `2424121224` de l'autre, et **aucun message d'erreur**. C'est
précisément ce qui vous attend en séance 2.2, où le prix arrive écrit
`2,08 EUR`. D'où le réflexe : après un `read_csv`, regarder les types avec
`info()` **avant** de calculer quoi que ce soit.

### Travailler volontairement sur du texte — `.str`

Pour manipuler du texte, pandas range ses outils derrière `.str`. Là encore,
toute la colonne est traitée d'un coup.

In [ ]:
print(pays.str.upper().head(2).tolist())   ## tout en majuscules
print(pays.str.len().head(3).tolist())     ## longueur de chaque chaine

# Des Vrai/Faux, comme au paragraphe 5 : .sum() les compte
print(pays.str.startswith("F").sum(), "clients dans un pays en F")

Les six commandes de texte à connaître :

| Commande | Effet |
|---|---|
| `.str.lower()` / `.str.upper()` | tout en minuscules / en majuscules |
| `.str.strip()` | enlève les espaces au début et à la fin |
| `.str.len()` | la longueur de chaque chaîne |
| `.str.replace("a", "b")` | remplace un morceau de texte |
| `.str.contains("Pays")` | vrai si la chaîne contient ce texte |
| `.str.startswith("F")` | vrai si elle commence par ce texte |

> 💡 Elles reviendront toutes en séance 2.2 : c'est avec elles qu'on répare
> une colonne de texte mal saisie.

---

### ✏️ Exercice — Le chiffre d'affaires total

> **Votre mission :**
> - Calculer le chiffre d'affaires **total** de l'année → `ca_total`, arrondi à 2 décimales.
> - La colonne `ca` existe déjà : il ne reste qu'à la sommer.

In [ ]:
# sum() additionne toute la colonne, ligne par ligne
ca_total = round(ventes["ca"].sum(), 2)

print("CA total :", ca_total, "euros")

In [ ]:
verifier("chiffre d'affaires total", ca_total == 1152913.87,
         "sum() sur la colonne ca, puis round(..., 2)")

---

### ✏️ Exercice — La plus grosse ligne du fichier

> **Votre mission :**
> - Retrouver la **ligne entière** dont le chiffre d'affaires est le plus élevé → `ligne_max`.
> - Mettre son `prod_id` dans `prod_max`, puis chercher ce `prod_id` dans `produits`.
> - Est-ce vraiment un produit ?
> - *Nouveau :* `df['ca'].idxmax()` donne l'**étiquette** de la ligne du maximum ; `df.loc[...]` va ensuite la chercher.

In [ ]:
ligne_max = ventes.loc[ventes["ca"].idxmax()]   ## idxmax = l'etiquette
prod_max = ligne_max["prod_id"]
print(ligne_max)

# prod_id vaut "M" : 4 161 EUR sur une seule ligne, et le catalogue ne
# le connait pas. "M" veut dire Manual, une saisie manuelle de facturation.
produits.query("prod_id == 'M'")

In [ ]:
verifier("la plus grosse ligne", prod_max == "M",
         "idxmax() sur la colonne ca, puis .loc pour aller chercher la ligne")

In [ ]:
ventes["Prix"]   ## erreur volontaire : la colonne est "prix"

In [ ]:
# Le reflexe quand on ne se souvient plus d'un nom de colonne
ventes.columns   ## la liste exacte, majuscules comprises

## 8. Nettoyer une base de données

`ventes.csv` était **impeccable** : pas un trou, pas un doublon, des types
corrects. Ça n'arrive jamais.

Voici le même détaillant, mais l'export tel qu'il sort vraiment du système :
`ventes_sale.csv`.

> 🎯 **Votre mission désormais:** transformer le fichier de données sales en données
> exploitables, et savoir dire **combien de lignes** vous avez perdues au
> passage et **pourquoi**.

> 📋 **Comment on travaille.** Cinq défauts. Pour chacun : une démonstration
> sur une table appelée `propre`, puis **deux exercices** où vous nettoyez la
> vôtre, appelée `net` — un à trous, un que vous écrivez en entier. À la fin,
> le fichier propre, c'est vous qui l'aurez construit.

In [ ]:
# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://cdn.jsdelivr.net/gh/maxischa/datacamp_test@5a33b79/bloc2_donnees/data/"

In [ ]:
sale = pd.read_csv(BASE + "ventes_sale.csv")   ## le fichier brut

print(sale.shape)   ## (lignes, colonnes)
sale.head(5)        ## cinq lignes suffisent a reperer l'essentiel

Prenez 30 secondes pour regarder ces cinq lignes. Qu'est-ce qui cloche ?

In [ ]:
sale.info()   ## regarder surtout la colonne Dtype

### Le diagnostic

`info()` révèle déjà deux problèmes graves :

- **`prix` est de type `object`** — c'est du **texte**, pas un nombre. On ne
  peut donc rien calculer avec. Les coupables sont visibles dès les cinq
  premières lignes : le suffixe de `0,42 EUR`, et la **virgule** décimale de
  `3,75` là où Python attend un point.
- **`date` est de type `object`** — du texte aussi. Impossible de demander
  « quel mois ? ».

Et un troisième, visible sur le `non-null` :

- **`client_id` a des trous.**

Il y en a quatre autres qu'`info()` ne montre pas. On va les débusquer.

## Défaut 1 — Les doublons

**On commence toujours par là**, avant toute autre étape de nettoyage.

La raison est un problème de comptage. Imaginez que vous commenciez par
retirer les ventes sans client : vous notez « 407 lignes retirées ». Mais
parmi ces 407, certaines étaient des copies l'une de l'autre. Vous n'avez donc
pas retiré 407 ventes, vous en avez retiré moins — et vous ne saurez jamais
combien. En dédoublonnant d'abord, chaque ligne du fichier est une vente
distincte, et tous les comptes qui suivent veulent dire quelque chose.

`duplicated()` ne rend pas un nombre. Comme la comparaison `prix > 50` de la
séance 2.1, il pose une question **à chaque ligne** — « celle-ci, je l'ai déjà
vue plus haut ? » — et rend **une réponse `True` ou `False` par ligne**, soit
5 370 réponses ici. Pour en tirer un nombre, on compte les `True` avec
`.sum()`.

In [ ]:
marques = sale.duplicated()   ## True = ligne deja vue plus haut

print(marques.iloc[140:145])   ## la 143e est une copie d'une ligne d'avant
print("lignes strictement identiques :", marques.sum())   ## .sum() compte les True

propre = sale.drop_duplicates().copy()
print(sale.shape[0], "->", propre.shape[0], "apres suppression des doublons")

> ⚠️ Nuance importante : ici les lignes sont **strictement identiques sur
> toutes les colonnes**, donc on peut supprimer. Si seul le `cmd_id` était en
> double, ce pourrait être un vrai client commandant deux fois le même
> article. Vérifiez toujours **sur quelles colonnes** porte le doublon avant
> de supprimer.

## Défaut 2 — Les valeurs manquantes

La commande à taper devant n'importe quel fichier :

In [ ]:
propre.isna().sum()   ## un compte de trous, colonne par colonne

407 lignes sans `client_id`. **Que faire ?**

Il n'y a pas de réponse universelle. Il y a une question à se poser :
**pourquoi cette valeur manque-t-elle ?**

Ici, probablement des ventes sans compte client (achat en magasin, commande
invitée). Donc :

| Votre question | La bonne décision |
|---|---|
| « Combien mes clients dépensent-ils ? » | **Supprimer** ces lignes : elles n'ont pas de client |
| « Quel est mon chiffre d'affaires total ? » | **Les garder** : ce sont de vraies ventes, les retirer fausserait le total |

### À quoi ressemblerait `fillna`, concrètement

`fillna(valeur)` remplace chaque trou par la valeur qu'on lui donne. Sur
cette colonne, ça s'écrirait comme ceci — regardez le résultat avant de
trouver la commande pratique.

In [ ]:
# On ecrit dans une colonne A COTE, jamais par-dessus l'originale
propre["client_id_new"] = propre["client_id"].fillna(0)   ## 0 dans les trous

print("trous restants :", propre["client_id_new"].isna().sum())
print(propre["client_id_new"].value_counts().head(3))

Plus un seul trou : mission accomplie ? Regardez le classement. Le « client
0 » arrive **deuxième du fichier** avec 407 achats, derrière un seul client
réel. Sauf que ce client n'existe pas : ce sont 407 acheteurs différents
regroupés sous une étiquette inventée. Toute analyse par client sera fausse,
et absolument rien ne vous préviendra.

> ⚠️ **Le piège à ne jamais commettre :** `fillna(0)` sur un identifiant.
> Remplir une valeur manquante, c'est **inventer une donnée** — ne le faites
> que si vous pouvez le justifier.

`fillna` a pourtant des usages parfaitement légitimes : une quantité absente
qu'on sait valoir 0, un libellé vide qu'on remplace par `"inconnu"`, un prix
manquant qu'on remplace par la médiane de sa catégorie. La question n'est
jamais « est-ce que ça marche ? » mais « qu'est-ce que j'affirme en
remplissant ce trou ? ».

In [ ]:
# Celle-la ne nous sert a rien : on la retire avant de continuer
propre = propre.drop(columns=["client_id_new"])   ## drop(columns=[...])

In [ ]:
# Notre question portera sur les clients : on supprime ces lignes,
# mais on note combien on en perd.
avant = len(propre)
propre = propre.dropna(subset=["client_id"]).copy()   ## cette colonne seule

print(avant, "->", len(propre), f"({avant - len(propre)} lignes retirees)")

## Défaut 3 — Des nombres stockés en texte

C'est le défaut le plus courant, et le plus sournois.

In [ ]:
propre["prix"].head(4)   ## du texte, pas des nombres

Deux problèmes dans une seule colonne :

1. Le suffixe **` EUR`** sur certaines valeurs.
2. La **virgule** comme séparateur décimal — convention française, alors que
   Python attend un point.

Trois étapes, dans cet ordre :

In [ ]:
# .str donne acces aux operations sur du texte, colonne entiere d'un coup
prix_txt = propre["prix"].str.replace(" EUR", "")   ## 1. l'unite
prix_txt = prix_txt.str.replace(",", ".")           ## 2. la virgule

propre["prix"] = pd.to_numeric(prix_txt, errors="coerce")   ## 3. la conversion
propre["prix"].head(4)   ## le type a change : ce sont des nombres

**`errors="coerce"`** veut dire : *« si tu n'arrives pas à convertir une
valeur, mets `NaN` au lieu de tout faire planter »*. C'est très pratique —
et très dangereux si on ne vérifie pas ensuite.

In [ ]:
# Reflexe obligatoire apres un coerce : combien de valeurs ont ete perdues ?
print("prix non convertis :", propre["prix"].isna().sum())

Zéro. Notre conversion est propre. **Faites systématiquement cette
vérification** : sans elle, vous pourriez transformer silencieusement 3 000
prix en `NaN` et ne vous en apercevoir qu'en présentant vos résultats.

## Défaut 4 — Les dates

Le plus piégeux. On procède en trois temps — mais d'abord, une question de
méthode.

### Temps 0 : savoir ce qu'on attend

On ne peut pas juger si une conversion a réussi sans savoir à quoi devrait
ressembler le résultat. La colonne est encore du texte, mais du texte régulier :
`24/11/2011`, `24-11-2011`. Ses **quatre derniers caractères** sont donc
l'année, et ses **troisième et quatrième** le mois, quel que soit le séparateur.
Cela suffit à cadrer la période sans rien convertir.

In [ ]:
# .str[3:5] et .str[-4:] : on decoupe le texte par position
annee = propre["date"].str[-4:]    ## les 4 derniers caracteres
mois = propre["date"].str[3:5]     ## les 3e et 4e

print(annee.value_counts().to_dict())
print("mois presents en 2010 :", sorted(mois[annee == "2010"].unique()))

Deux années : 335 lignes en 2010 et 4 377 en 2011. Et en 2010, **un seul
mois** — décembre. Ce fichier couvre donc **de décembre 2010 à décembre
2011**, ce qui est cohérent avec l'historique d'un an qu'on nous a annoncé.

Retenez ce repère : c'est lui qui va nous permettre, dans deux cellules, de
repérer une conversion qui a échoué sans le dire.

**Temps 1 :** la façon naïve.

In [ ]:
# Cellule volontairement fausse : lisez le message d'erreur
pd.to_datetime(propre["date"])   ## sans format, pandas devine... et echoue

`ValueError: time data "24-11-2011" doesn't match format "%d/%m/%Y"`

Le fichier mélange deux écritures : `14/11/2011` et `24-11-2011`. pandas veut
un format unique. Le message suggère lui-même la solution : `format="mixed"`.

**Temps 2 :** on ajoute `format="mixed"`.

In [ ]:
essai = pd.to_datetime(propre["date"], format="mixed")   ## deux ecritures

print("date la plus ancienne :", essai.min())
print("date la plus recente  :", essai.max())

Plus d'erreur. Mais confrontez le résultat au repère du temps 0 : le fichier
va de décembre 2010 à décembre 2011, et pandas annonce **janvier 2010**.
**C'est faux.**

Pourquoi ? Parce que `01/12/2010` a été lu **à l'américaine** : mois d'abord,
donc le 12 janvier. En français, c'est le 1er décembre.

**Temps 3 :** on impose la lecture française avec `dayfirst=True`.

In [ ]:
# dayfirst=True : lecture francaise, le jour avant le mois
propre["date"] = pd.to_datetime(propre["date"], format="mixed", dayfirst=True)

print("date la plus ancienne :", propre["date"].min())
print("date la plus recente  :", propre["date"].max())

> ⚠️ **Le point le plus important de la séance.** L'étape 1 produisait une
> **erreur bruyante** : gênante, mais elle vous arrête. L'étape 2 produisait
> une **erreur silencieuse** : le code tourne, les chiffres s'affichent, et
> ils sont faux. C'est de très loin la plus dangereuse.
>
> Après toute conversion, **vérifiez que le résultat est plausible** :
> `.min()`, `.max()`, un `head()`. Trente secondes qui vous éviteront de
> présenter des chiffres faux.

Une fois la colonne convertie en date, `.dt` ouvre tout :

In [ ]:
propre["mois"] = propre["date"].dt.month           ## .dt = boite a outils
propre["jour_sem"] = propre["date"].dt.dayofweek   ## 0 = lundi, 6 = dimanche

propre[["date", "mois", "jour_sem"]].head(3)

## Défaut 5 — Du texte incohérent

In [ ]:
print("nombre de categories distinctes :", propre["categorie"].nunique())
propre["categorie"].unique()[:8]

24 catégories, alors qu'il n'en existe que 8. Regardez bien : `' cuisine'`
avec un espace devant, `'CUISINE'` en majuscules, `'cuisine'`. Pour pandas,
ce sont **trois catégories différentes** — et un `value_counts()` sur cette
colonne éclaterait la cuisine en trois lignes, chacune sous-estimée.

> ⚠️ L'espace en début de chaîne est **invisible à l'écran**. C'est ce qui
> rend ce défaut particulièrement traître.

In [ ]:
# .str.strip() enleve les espaces au bord, .str.lower() met en minuscules
propre["categorie"] = propre["categorie"].str.strip().str.lower()   ## 24 -> 8

print("apres nettoyage :", propre["categorie"].nunique(), "categories")

## Défaut 6 — Les valeurs aberrantes

In [ ]:
propre["qte"].describe().round(1)   ## regarder min et max avant tout

Un minimum **négatif** et un maximum à **99 999**. Deux anomalies, mais elles
n'ont rien à voir :

- **`qte` négatif** : ce sont des **retours**. Ce n'est pas une erreur, c'est
  une information métier. On les écarte du calcul de chiffre d'affaires, mais
  on ne les jette pas — un taux de retour, ça s'analyse.
- **`qte = 99999`** : personne ne commande 99 999 articles. C'est une saisie
  erronée, ou un code sentinelle. On l'écarte.

> Traiter ces deux cas de la même façon serait une faute d'analyse.

In [ ]:
retours = propre.query("qte < 0")   ## un retour, pas une erreur
print("retours :", len(retours), "lignes")
print("quantites aberrantes :", len(propre.query("qte >= 10000")), "lignes")

avant = len(propre)
propre = propre.query("qte > 0 and qte < 10000").copy()   ## "and" dans query
print(avant, "->", len(propre))

## Enrichir — classer des lignes selon une règle

Une dernière commande, qui ne répare rien : elle **ajoute** une colonne. On
écrit la règle **une fois**, pandas l'applique à chaque ligne.

In [ ]:
propre["ca"] = propre["qte"] * propre["prix"]   ## calculable enfin

# np.where(condition, valeur_si_vrai, valeur_si_faux)
propre["type"] = np.where(propre["ca"] > 50, "grosse", "petite")   ## deux cas
propre["type"].value_counts()

> 💡 `np.where` ne tranche qu'entre **deux** possibilités. Pour trois catégories
> ou plus, et pour découper une colonne en tranches, deux commandes vous
> attendent dans la **feuille facultative** : `np.select` et `pd.cut`. Elles
> resserviront au bloc 3.

## Le bilan

Bonne pratique pour finir : un petit compte rendu de ce qu'on a retiré. Ça
tient en trois lignes, et ça évite d'avoir à se demander, trois semaines plus
tard, d'où viennent les lignes qui manquent.

---

# Corrigé de la feuille

Les exercices de la séance sont corrigés plus haut, dans le fil du cours. Les cellules ci-dessous rejouent le setup pour rester exécutables isolément.

## Exercice — Où acheter à Paris ?

## La question

Vous êtes analyste dans une agence immobilière parisienne. Un client dispose de
**400 000 €** et vous pose une question simple :

> *« Dans quel arrondissement est-ce que j'achète le plus de mètres carrés ? »*

Pour répondre, l'agence vous remet l'export brut de **toutes les ventes
immobilières enregistrées à Paris en 2024**. Ce sont des données publiques,
publiées par l'administration fiscale, et elles sont dans l'état où on les
reçoit : personne ne les a nettoyées avant vous.

À la fin de cet exercice, vous aurez :

- un fichier propre, dont vous saurez dire ligne par ligne ce que vous avez retiré et pourquoi ;
- le prix au mètre carré de chacun des vingt arrondissements ;
- la réponse au client, chiffrée ;
- et une carte de Paris colorée par le prix, que vous aurez produite vous-même.

## Comment ça marche

L'exercice se fait seul, en quatre parties, dans l'ordre. Comptez une demi-heure
pour la première, une heure et demie pour la deuxième, une heure pour la
troisième, une demi-heure pour la dernière.

**Chaque exercice est une cellule vide que vous écrivez entièrement.** Juste
avant, un encadré *Rappel* vous redonne la forme de la commande vue en cours.
Vous n'avez rien à deviner : vous avez à l'appliquer à ce fichier.

Vous rencontrerez aussi trois autres sortes de cellules :

- des **cellules à exécuter telles quelles** : le code est déjà écrit, il vous montre quelque chose ;
- des **cellules de vérification**, aux moments où une erreur fausserait toute la suite. Elles affichent `OK` ou `A REVOIR` avec un indice ;
- des **cellules de prédiction** : on vous demande d'écrire ce que vous attendez, en commentaire, *avant* d'exécuter la cellule suivante. Prenez-les au sérieux, c'est là qu'on apprend.

Si une vérification affiche `NameError`, c'est que la cellule d'exercice
au-dessus n'a pas été exécutée, ou qu'elle contient une faute. Corrigez-la,
relancez-la, puis relancez la vérification.

Tout ce dont vous avez besoin est dans la séance pandas et dans la séance 1.1.
Quand un outil nouveau est nécessaire, il est présenté sur place.

## Partie 0 — Mise en route

La cellule de setup est la même que dans tous les notebooks du cours. Exécutez-la
en premier.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://cdn.jsdelivr.net/gh/maxischa/datacamp_test@5a33b79/bloc2_donnees_v2/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

In [ ]:
brut = pd.read_csv(BASE + "immo_paris_sale.csv")   ## le fichier brut, tel que recu

---

## Partie 1 — Découvrir le fichier

Avant de nettoyer quoi que ce soit, il faut savoir ce qu'on a entre les mains.
Le réflexe du cours, toujours dans cet ordre : `shape`, `info()`, `head()`,
`describe()`. Trente secondes, et vous savez de quoi vous parlez.

### Exercice 1 — La carte d'identité du fichier

Affichez la taille du tableau, ses colonnes avec leur type, et ses cinq
premières lignes. Puis, en commentaire à la fin de la cellule, répondez à la
question : **quelles colonnes devraient contenir des nombres, et n'en
contiennent pas ?** Regardez la colonne `Dtype` de `info()` : `object` veut
dire texte.

> **Rappel.** Dans une même cellule, seule la dernière expression s'affiche
> toute seule : mettez les autres dans un `print()`, ou une par cellule.

In [ ]:
print(brut.shape)
brut.info()
brut.head()

# prix et date sont de type object (du texte) alors que ce sont un montant et une date

Dix colonnes. Vous les retrouverez tout au long de l'exercice :

| Colonne | Contenu |
|---|---|
| `vente_id` | l'identifiant de la vente |
| `date` | la date de la vente |
| `prix` | le prix payé, en euros |
| `rue` | le nom de la rue |
| `arrondissement` | de 1 à 20 |
| `type_local` | ce qui est vendu : appartement, dépendance, local commercial, maison |
| `surface` | la surface bâtie, en m² |
| `pieces` | le nombre de pièces principales |
| `longitude`, `latitude` | la position sur la carte |

### Exercice 2 — La photo chiffrée

`describe()` donne pour chaque colonne de nombres ses huit statistiques :
effectif, moyenne, écart-type, minimum, les trois quartiles et maximum.
Affichez-le, arrondi à une décimale, puis répondez aux deux questions en
commentaire :

1. Une colonne que vous attendiez n'apparaît pas dans le tableau. Laquelle, et
   pourquoi ?
2. Regardez le `max` de `surface`. Est-ce une surface d'appartement ?

In [ ]:
brut.describe().round(1)

# 1. prix n'apparait pas : describe() ne traite que les colonnes de nombres, et prix est du texte
# 2. non : plus de 48 000 m2, c'est un entrepot ou un immeuble entier, pas un appartement

Deux choses à retenir de ce tableau.

`prix` n'y est pas, et **pandas ne s'en est pas plaint**. `describe()` ne
traite que les colonnes de nombres ; une colonne de texte est simplement
laissée de côté, sans message.

`surface` a un maximum de plusieurs dizaines de milliers de m². Personne
n'habite cinq hectares : le fichier ne contient donc pas que des appartements.

### Exercice 3 — Une ligne, c'est quoi ?

C'est la question à se poser devant n'importe quel fichier. Ici, une
ligne est-elle une vente ? Pour le savoir, comparez le nombre de lignes au
nombre de valeurs **distinctes** de `vente_id`.

> **Rappel.** `len` ou `shape` comptent les lignes.
> `nunique` compte les valeurs distinctes.

In [ ]:
nb_lignes = len(brut)
nb_ventes = brut["vente_id"].nunique()

print(nb_lignes, "lignes pour", nb_ventes, "ventes distinctes")

In [ ]:
verifier("nombre de lignes", nb_lignes == 61276, "len(brut)")
verifier("nombre de ventes distinctes", nb_ventes == 35631, "nunique() sur la colonne vente_id")

Presque deux fois plus de lignes que de ventes. Une ligne n'est donc pas une
vente. Regardons une vente précise pour comprendre. Exécutez la cellule telle
quelle :

In [ ]:
brut.query("vente_id == '2024-1193275'")   ## une seule vente, toutes ses lignes

Une vente, trois lignes : un appartement, et deux dépendances (une cave, un
parking, ce genre de lots). **Chaque ligne est un lot**, et une vente peut en comporter
plusieurs. Le `prix` est le même sur les trois lignes : c'est le prix de la
vente entière, pas celui du lot.

Retenez-le, c'est le piège principal de ce fichier. Pour calculer un prix au
mètre carré d'appartement, il faudra isoler les ventes qui ne contiennent
qu'un seul appartement. Ce sera la dernière étape du nettoyage.

Remarquez aussi comment `type_local` est écrit d'une ligne à l'autre.

### Exercice 4 — Compter les types de biens

Comptez les valeurs de `type_local` avec `value_counts()`, après mettez le nombre
d'écritures différentes dans `nb_ecritures`. Combien de types de biens existent
**vraiment** ?

> **Rappel.** `value_counts` pour avoir les effectifs par catégorie, sous la forme
> d'un tableau trié. Le nombre de lignes de ce tableau donne le nombre d'écritures.

In [ ]:
types = brut["type_local"].value_counts()
nb_ecritures = len(types)

print(nb_ecritures, "ecritures differentes")
types

In [ ]:
verifier("nombre d'ecritures", nb_ecritures == 16, "len() sur le resultat de value_counts()")

Seize écritures pour quatre types réels : `Appartement`, `Dépendance`, `Local
industriel. commercial ou assimilé`, `Maison`. Les autres sont les mêmes mots
avec un espace devant, un espace derrière, ou en majuscules. Pour pandas, ce
sont seize catégories différentes. On réparera ça en partie 2.

### Exercice 5 — Le prix moyen

Calculez le prix moyen des ventes du fichier.

**Ça va échouer.** Exécutez quand même, puis lisez la dernière ligne du message
et écrivez en commentaire, dans la cellule d'après, pourquoi ça ne marche pas.
Vous avez tous les éléments depuis l'exercice 1.

In [ ]:
brut["prix"].mean()

In [ ]:
# prix est du texte : on ne peut pas faire la moyenne de mots. Il faut d'abord
# le convertir en nombre. C'est tout l'objet de la partie 2.

`TypeError`, et le message parle de texte (`string`) qu'il ne peut pas
convertir en nombre.

Vous ne pouvez rien calculer sur ce fichier tant qu'il n'est pas nettoyé. C'est
le programme de la partie 2.

---

## Partie 2 — Nettoyer

Sept défauts, un par section, toujours traités de la même façon :

1. **constater** : compter le défaut ;
2. **comprendre** : d'où vient-il, et qu'est-ce que ça change ;
3. **corriger** ;
4. **mesurer** : combien de lignes ont disparu, noté dans un journal ;
5. **vérifier**.

Le fichier nettoyé s'appelle `propre`. Le fichier brut reste `brut`, on ne le
modifie jamais : si quelque chose se passe mal, on repart de lui.

Le journal est une liste, comme à la séance 1.1. On y ajoute une phrase à chaque
étape, et on l'imprimera à la fin.

In [ ]:
journal = []   ## une phrase par etape de nettoyage

### Défaut 1 — Les doublons

On commence toujours par là. Tant qu'une ligne peut être la copie d'une autre,
aucun compte ne veut rien dire : on croirait retirer 100 ventes sans client
alors qu'on en retire 80 et 20 copies.

### Exercice 6 — Dédoublonner

Comptez les lignes strictement identiques à une ligne précédente, mettez le
résultat dans `nb_doublons`, puis créez `propre` : le fichier brut sans ces
copies. Ajoutez une phrase au journal.

> **Rappel.** `df.duplicated()` rend un Vrai/Faux par ligne, `.sum()` compte
> les Vrai. `df.drop_duplicates()` retire les copies. Ajoutez `.copy()` après
> un filtrage, pour avoir un vrai tableau à soi.
>
> **Rappel.** `journal.append(f"doublons : {nb_doublons} lignes")` ajoute une
> phrase à la liste.

In [ ]:
nb_doublons = brut.duplicated().sum()
propre = brut.drop_duplicates().copy()

journal.append(f"doublons exacts : {nb_doublons} lignes")
print(len(brut), "->", len(propre))

In [ ]:
verifier("doublons retires", len(propre) == 59492, "drop_duplicates() sur brut, puis .copy()")

### Défaut 2 — Les valeurs manquantes

### Exercice 7 — Compter les trous

Comptez les valeurs manquantes de chaque colonne de `propre`, et mettez le
résultat dans `trous`.

> **Rappel.** `df.isna().sum()` : un compte de trous par colonne.

In [ ]:
trous = propre.isna().sum()
trous

In [ ]:
verifier("surfaces manquantes", trous["surface"] == 24297, "isna().sum() sur propre, puis la ligne surface")

Plus de 24 000 surfaces manquantes, soit **40 % du fichier**. Le réflexe
serait de les supprimer. Ce serait une erreur de méthode.

La question du cours : **pourquoi cette valeur manque-t-elle ?** On ne
supprime pas un trou sans savoir ce qu'il y a derrière. La cellule suivante
marque les lignes sans surface dans une colonne, puis n'affiche qu'elles.
Exécutez-la telle quelle et regardez la colonne `type_local`.

In [ ]:
propre["sans_surface"] = propre["surface"].isna()      ## Vrai si la surface manque
propre.query("sans_surface == True").head(8)           ## on ne garde que ces lignes

Des dépendances, sous leurs différentes écritures. Une cave ou un parking n'a
pas de surface habitable : **le trou est normal**, il ne signale aucune erreur.

Et surtout, ces lignes ne nous intéressent pas. Notre question porte sur des
appartements. Si on supprimait les surfaces manquantes maintenant, on
retirerait les bonnes lignes pour la mauvaise raison, et le journal dirait
« 24 000 lignes perdues pour cause de données manquantes », ce qui est faux.

**On ne supprime rien ici.** On traitera les types de biens au défaut 5, et on
reviendra alors sur les trous qui restent. La colonne de marquage ne sert
plus, on la retire :

In [ ]:
propre = propre.drop(columns=["sans_surface"])   ## drop(columns=[...]) retire une colonne

### Défaut 3 — Les nombres stockés en texte

Regardez à quoi ressemble la colonne `prix` :

In [ ]:
propre["prix"].head(8)

Deux problèmes, comme en cours, mais pas tout à fait les mêmes : une
**virgule** décimale là où Python attend un point, et un suffixe sur certaines
valeurs. Regardez bien lequel : ce n'est pas celui du fichier de cours.

### Exercice 8 — Le prix en nombre

Avant d'écrire, une prédiction. À l'exercice 7, `trous` vous a dit combien de
prix manquaient déjà dans le fichier. Une conversion réussie ne doit en perdre
**aucun de plus**. Combien de `NaN` attendez-vous après conversion ?

In [ ]:
# Ma prediction (nombre de NaN attendus apres conversion) :

Maintenant convertissez : retirez le suffixe, remplacez la virgule par un
point, convertissez en nombre, remettez le résultat dans `propre["prix"]`. Puis
comptez les `NaN` obtenus dans `nb_prix_perdus`.

> **Rappel.** Les trois étapes du cours :
> ```python
> txt = df["prix"].str.replace(" EUR", "")     # 1. le suffixe (adaptez-le !)
> txt = txt.str.replace(",", ".")              # 2. la virgule
> df["prix"] = pd.to_numeric(txt, errors="coerce")   # 3. la conversion
> ```
> `errors="coerce"` met `NaN` quand il ne sait pas convertir, sans prévenir.
> D'où le compte obligatoire juste après.

In [ ]:
txt = propre["prix"].str.replace(" €", "")
txt = txt.str.replace(",", ".")
propre["prix"] = pd.to_numeric(txt, errors="coerce")

nb_prix_perdus = propre["prix"].isna().sum()
print(propre["prix"].dtype, "|", nb_prix_perdus, "prix manquants apres conversion")

In [ ]:
verifier("prix en nombres", propre["prix"].dtype == "float64", "pd.to_numeric sur le texte nettoye")
verifier("aucun prix perdu", nb_prix_perdus == 79,
         "si vous en avez pres de 8937, le suffixe n'a pas ete retire : regardez-le dans les donnees, ce n'est pas ' EUR'")

Si votre vérification a échoué avec près de 9 000 prix perdus, vous venez de
vivre le danger de `errors="coerce"` : **aucune erreur, aucun message, et 15 %
des prix disparus**. Corrigez le suffixe, relancez, recomptez. Ce compte après
conversion n'est pas une option.

### Défaut 4 — Les dates

Le défaut le plus piégeux du cours, et ici le piège est plus dur. On procède
comme en cours, en quatre temps.

**Temps 0 : se donner un repère avant de convertir.** La colonne est du texte
régulier : `16/02/2024`, `12.12.2024`. Quel que soit le séparateur, les
caractères 3 et 4 sont le mois. On peut donc compter les ventes par mois
**sans rien convertir**, et garder ce compte comme référence.

In [ ]:
mois_texte = propre["date"].str[3:5].value_counts().sort_index()   ## le mois, lu dans le texte
mois_texte

Douze mois, et un creux net en août. C'est notre repère : après conversion,
on doit retrouver **exactement** ces nombres.

**Temps 1 : la façon naïve.** Cellule volontairement fausse, lisez le message.

In [ ]:
pd.to_datetime(propre["date"])   ## sans format, pandas devine... et echoue

Le fichier mélange deux écritures et pandas veut un format unique. Le message
suggère lui-même `format="mixed"`.

**Temps 2 : avec `format="mixed"`.** Plus d'erreur. Regardons si le résultat
est plausible, avec la vérification habituelle du cours, puis avec notre
repère.

In [ ]:
essai = pd.to_datetime(propre["date"], format="mixed")   ## sans dayfirst

print("plus ancienne :", essai.min().date(), "| plus recente :", essai.max().date())

# Notre repere : les ventes par mois, lues dans le texte contre lues apres conversion
comparaison = pd.DataFrame({"dans le texte": mois_texte.values,
                            "apres conversion": essai.dt.month.value_counts().sort_index().values},
                           index=range(1, 13))
comparaison

Le minimum et le maximum sont parfaitement plausibles : de janvier à décembre
2024. En cours, cette vérification suffisait. **Ici, elle ne voit rien.**

Mais la comparaison avec le repère, elle, est sans appel : les nombres ne
correspondent pas, et août a gagné plusieurs centaines de ventes. Un tiers des
dates ont changé de mois. `12/03/2024` a été lu à l'américaine, mois d'abord :
le 3 décembre au lieu du 12 mars.

La leçon dépasse ce fichier : **une conversion qui ne produit pas d'erreur
n'est pas une conversion réussie**, et le minimum et le maximum ne suffisent
pas toujours à le voir. Il faut s'être donné un repère avant.

### Exercice 9 — Les dates, correctement

**Temps 3.** Convertissez `propre["date"]` en vraies dates, en imposant la
lecture française. Puis recomptez les ventes par mois à partir des dates
converties, mettez ce compte dans `mois`, et le nombre de ventes d'août dans
`nb_aout`. Il doit être égal à celui du repère.

> **Rappel.** `pd.to_datetime(col, format="mixed", dayfirst=True)`. Une fois la
> colonne convertie, `.dt.month` donne le mois de chaque ligne.
>
> **Sur une colonne de comptes.** `mois.loc[8]` va chercher la valeur
> dont l'étiquette est 8. C'est le même `.loc` que sur un tableau : on donne
> l'étiquette, il rend la valeur.

In [ ]:
propre["date"] = pd.to_datetime(propre["date"], format="mixed", dayfirst=True)

mois = propre["date"].dt.month.value_counts().sort_index()
nb_aout = mois.loc[8]

print(nb_aout, "ventes en aout ; le repere disait", mois_texte.loc["08"])

In [ ]:
verifier("dates converties", str(propre["date"].dtype).startswith("datetime"), "pd.to_datetime avec format='mixed'")
verifier("aout retrouve", nb_aout == 2257, "sans dayfirst=True, aout affiche 3102 : le jour et le mois sont inverses")

### Défaut 5 — Du texte incohérent

Retour sur `type_local` et ses seize écritures pour quatre types. Un espace
devant, un espace derrière, des majuscules : trois défauts que deux commandes
réparent.

### Exercice 10 — Uniformiser les types

Enlevez les espaces autour et passez tout en minuscules, dans
`propre["type_local"]`. Mettez le nombre de types restants dans `nb_types`.

> **Rappel.** `str.strip()` retire les espaces, `str.lower()` met la chaîne de caractères en minuscules.
> `nunique()` compte le nombre de valeurs uniques.

In [ ]:
propre["type_local"] = propre["type_local"].str.strip().str.lower()
nb_types = propre["type_local"].nunique()

print(nb_types, "types")
propre["type_local"].value_counts()

In [ ]:
verifier("quatre types", nb_types == 4, "strip() enleve les espaces des deux cotes, lower() la casse : il faut les deux")

### Exercice 11 — Ne garder que les appartements

Notre question porte sur des appartements. Les caves, les parkings, les
boutiques et les maisons sortent du périmètre.

Gardez dans `propre` les seules lignes dont le type est `appartement`. Mettez
le nombre de lignes obtenu dans `nb_app` : ce nombre servira au bilan. Notez
au journal les lignes écartées, en précisant bien qu'il s'agit d'un choix de
périmètre.

> **Rappel.** `df.query(condition)`. La condition, avec des guillemets doubles
> dehors, simples dedans.

In [ ]:
avant = len(propre)
propre = propre.query("type_local == 'appartement'").copy()
nb_app = len(propre)

journal.append(f"hors perimetre (pas des appartements) : {avant - nb_app} lignes")
print(avant, "->", nb_app)

In [ ]:
verifier("appartements seuls", nb_app == 30833, "query sur type_local == 'appartement', apres strip et lower")

### Exercice 12 — Retour sur les valeurs manquantes

On avait laissé les trous de côté au défaut 2, en promettant d'y revenir une
fois les dépendances écartées. Recomptez les valeurs manquantes de `propre`.
Il n'en reste qu'une poignée, sur `prix` et `surface` : ce sont de vraies
lacunes, sur des appartements. Sans prix ou sans surface, pas de prix au m² :
retirez ces lignes, et notez-les au journal.

Pourquoi ne pas les remplir, par la moyenne par exemple ? Parce qu'un prix au
m² calculé sur une surface inventée serait un chiffre inventé. Remplir un trou,
c'est affirmer quelque chose ; ici on n'a rien à affirmer.

> **Rappel.** `df.isna().sum()` : un compte de trous par colonne. `dropna(subset=["a", "b"])` retire les
> lignes où l'une de ces colonnes ("a" ou "b") est vide.

In [ ]:
print(propre.isna().sum())

avant = len(propre)
propre = propre.dropna(subset=["prix", "surface"]).copy()

journal.append(f"prix ou surface manquants : {avant - len(propre)} lignes")
print(avant, "->", len(propre))

In [ ]:
verifier("lignes completes", len(propre) == 30794, "dropna(subset=['prix', 'surface'])")

### Défaut 6 — Les valeurs aberrantes

Maintenant que `prix` est un nombre, `describe()` peut enfin le voir. Exécutez :

In [ ]:
propre[["prix", "surface"]].describe().round(0)

Lisez `min` et `max`. Un appartement vendu **1 €**, un autre plusieurs
centaines de millions ; une surface de **2 m²**, une autre de plus de 1 000 m².
Et regardez la moyenne du prix face à sa médiane : la moyenne est plusieurs
fois plus haute. C'est la signature d'une poignée de valeurs énormes qui la
tirent vers le haut.

Ces valeurs ne sont pas des fautes de frappe. Un euro, c'est une vente
symbolique entre membres d'une famille ; cent millions, c'est un immeuble
entier enregistré comme un seul lot ; deux mètres carrés, c'est une chambre de
service. Ce sont de vraies transactions, mais elles ne disent rien sur le prix
d'un appartement parisien. On les écarte, et on note qu'on l'a fait.

### Exercice 13 — Le prix au mètre carré

Créez la colonne `prix_m2` : le prix divisé par la surface. C'est elle qui
répondra au client.

> **Rappel.** `df["c"] = df["a"] / df["b"]` : une seule ligne, calculée pour
> toutes les lignes.

In [ ]:
propre["prix_m2"] = propre["prix"] / propre["surface"]
propre[["prix", "surface", "prix_m2"]].head()

Avant de fixer un seuil, regardez les extrêmes. `sort_values` trie un tableau
selon une colonne ; `head(3)` garde les trois premiers. Exécutez les deux
cellules :

In [ ]:
propre.sort_values("prix_m2").head(3)[["prix", "surface", "prix_m2", "arrondissement"]]   ## les moins chers

In [ ]:
propre.sort_values("prix_m2", ascending=False).head(3)[["prix", "surface", "prix_m2", "arrondissement"]]   ## les plus chers

### Exercice 14 — Borner

On garde les appartements d'au moins **9 m²** (c'est le minimum légal pour
louer un logement) et dont le prix au m² est compris entre **2 000 et 30 000 €**,
la fourchette du marché parisien. Tout le reste est écarté, et noté au journal.

> **Rappel.** Un encadrement s'écrit d'un seul tenant, et deux conditions se combinent avec `and`.

In [ ]:
avant = len(propre)
propre = propre.query("surface >= 9 and 2000 <= prix_m2 <= 30000").copy()

journal.append(f"prix ou surface aberrants : {avant - len(propre)} lignes")
print(avant, "->", len(propre))

In [ ]:
verifier("valeurs bornees", len(propre) == 26778, "surface >= 9 and 2000 <= prix_m2 <= 30000, dans une seule chaine")

### Défaut 7 — Un lot n'est pas une vente

Souvenez-vous de l'exercice 3 : une vente peut contenir plusieurs lots, et le
prix inscrit est celui de la vente entière. On a écarté les caves et les
parkings, mais il reste un cas : **une vente qui contient deux appartements**,
ou plus. Deux lignes, un seul prix pour les deux. Le `prix_m2` calculé sur
chacune est faux, et on n'a aucun moyen de répartir le prix entre elles.

La seule décision honnête : ne garder que les ventes qui ne contiennent qu'un
appartement.

**Un outil nouveau.** `df.duplicated(subset=["vente_id"])` compare les lignes
sur cette seule colonne, et marque la deuxième occurrence, la troisième, etc.
Ce n'est pas ce qu'on veut : on veut écarter **toutes** les lignes d'une vente
répétée, la première comprise. C'est ce que fait l'option `keep=False` : elle
marque toutes les occurrences.

```python
df.duplicated(subset=["vente_id"], keep=False)   # Vrai sur TOUTES les lignes d'une vente repetee
```

### Exercice 15 — Isoler les ventes d'un seul appartement

Marquez les lignes dont le `vente_id` apparaît plusieurs fois, dans une colonne
`multi`. Comptez-les dans `nb_multi`. Puis ne gardez que les lignes où `multi`
est faux, et notez au journal.

Pour finir, la vérification qui referme la question de l'exercice 3 : le nombre
de lignes de `propre` doit maintenant être **égal** au nombre de ventes
distinctes.

> **Rappel.** Marquer dans une colonne, puis filtrer avec
> `query("multi == False")`, comme au défaut 2.

In [ ]:
propre["multi"] = propre.duplicated(subset=["vente_id"], keep=False)
nb_multi = propre["multi"].sum()

avant = len(propre)
propre = propre.query("multi == False").copy()

journal.append(f"ventes de plusieurs appartements : {avant - len(propre)} lignes")
print(avant, "->", len(propre))
print("une ligne = une vente ?", len(propre) == propre["vente_id"].nunique())

In [ ]:
verifier("ventes d'un seul appartement", len(propre) == 25209, "duplicated(subset=['vente_id'], keep=False), puis garder multi == False")
verifier("une ligne = une vente", len(propre) == propre["vente_id"].nunique(), "il reste des ventes repetees")

### Le bilan

Le journal, imprimé par une boucle :

In [ ]:
for ligne in journal:
    print(ligne)

print()
print("fichier recu :", len(brut), "lignes | fichier propre :", len(propre), "lignes")

### Exercice 16 — Le taux de perte

Plus de la moitié des lignes ont disparu. Le cours dit qu'au-delà de 40 % de
pertes, il faut reprendre le pipeline. Faut-il le reprendre ?

Non, et il faut savoir l'expliquer. La plus grosse ligne du journal n'est pas
une perte : c'est un **choix de périmètre**. Les caves et les boutiques ne sont
pas des données abîmées, ce sont des données qui ne concernent pas la
question. Le vrai taux de perte se mesure **sur les appartements** : parmi les
`nb_app` appartements de l'exercice 11, combien ont été écartés parce qu'ils
étaient inexploitables ?

Calculez ce taux en pourcentage, arrondi à une décimale, dans `taux_perte`.

> **Rappel.** `100 * (1 - apres / avant)`, puis `round(..., 1)`.

In [ ]:
taux_perte = round(100 * (1 - len(propre) / nb_app), 1)

print("perte sur les appartements :", taux_perte, "%")

In [ ]:
verifier("taux de perte", taux_perte == 18.2, "comparez le fichier final a nb_app, pas au fichier brut")

Moins de 20 %, et chaque ligne écartée a sa raison dans le journal. C'est ce
que vous présenteriez à votre responsable : pas « j'ai nettoyé les données »,
mais « voici ce que j'ai retiré, et pourquoi ».

Vous avez fini la partie la plus longue. Le fichier est propre, et vous savez
exactement ce qu'il contient : **une ligne, un appartement, une vente**, avec
un prix au mètre carré fiable. Tout ce qui suit va vite.

---

## Partie 3 — Analyser et tracer

### Cellule de rattrapage

Si votre nettoyage n'a pas abouti, ou si vous avez un doute, exécutez la cellule
ci-dessous : elle refait tout le nettoyage d'un coup, dans l'ordre, et produit
le même `propre`. Si tout est allé bien, vous pouvez l'exécuter quand même, elle
ne change rien.

C'est aussi, en douze lignes, le résumé de la partie 2. Relisez-la : c'est un
pipeline, qu'on peut rejouer sur le fichier de l'année prochaine.

In [ ]:
propre = pd.read_csv(BASE + "immo_paris_sale.csv").drop_duplicates().copy()
propre["prix"] = pd.to_numeric(propre["prix"].str.replace(" €", "").str.replace(",", "."), errors="coerce")
propre["date"] = pd.to_datetime(propre["date"], format="mixed", dayfirst=True)
propre["type_local"] = propre["type_local"].str.strip().str.lower()
propre = propre.query("type_local == 'appartement'").copy()
propre = propre.dropna(subset=["prix", "surface"]).copy()
propre["prix_m2"] = propre["prix"] / propre["surface"]
propre = propre.query("surface >= 9 and 2000 <= prix_m2 <= 30000").copy()
propre["multi"] = propre.duplicated(subset=["vente_id"], keep=False)
propre = propre.query("multi == False").copy()

print(propre.shape)   ## (25209, 12)

### Exercice 17 — La répartition des prix au m²

Une moyenne donne un chiffre ; un histogramme donne la forme. Tracez
l'histogramme de `prix_m2` en 50 classes, avec un titre et l'unité en abscisse.

Puis, en commentaire : où se concentre le gros des ventes ? La répartition
est-elle symétrique, ou étirée d'un côté ?

> **Rappel.**
> ```python
> df["col"].plot(kind="hist", bins=50, figsize=(7, 4))
> plt.title("...")
> plt.xlabel("...")
> plt.show()
> ```

In [ ]:
propre["prix_m2"].plot(kind="hist", bins=50, figsize=(7, 4))
plt.title("Repartition des prix au m2 (Paris, appartements, 2024)")
plt.xlabel("Prix au m2 (euros)")
plt.show()

# Le gros des ventes est entre 7 000 et 13 000 euros le m2 ; la queue est etiree vers la droite

### Le prix médian par arrondissement

C'est le cœur de la réponse. Il faut une médiane par arrondissement, donc vingt
médianes. Vous savez calculer la médiane d'une colonne ; vous savez garder les
lignes d'un arrondissement avec `query`. Il reste à faire les deux **vingt
fois**, et à ranger les vingt résultats.

#### Une variable dans une condition

`query` travaille sur une chaîne de caractères. Pour y utiliser la valeur d'une
variable Python, on met `@` devant son nom :

In [ ]:
arr = 12
douze = propre.query("arrondissement == @arr")   ## @arr : la valeur de la variable arr

print(len(douze), "ventes dans le 12e, prix median", round(douze["prix_m2"].median()), "euros le m2")

Sans le `@`, pandas chercherait une colonne nommée `arr`. Avec, il lit la
variable. C'est ce qui permet d'écrire la ligne **une fois** et de la faire
tourner pour les vingt arrondissements.

#### La boucle

Le principe est celui de la séance 1.1 : une liste vide avant la boucle, qu'on
remplit avec `append` à chaque tour. Ici il en faut deux, une pour les numéros
d'arrondissement, une pour les médianes.

### Exercice 18 — Vingt médianes

Écrivez une boucle `for` sur `range(1, 21)`. À chaque tour : filtrez `propre` sur
l'arrondissement, calculez la médiane de `prix_m2`, et ajoutez le numéro à la
liste `numeros` et la médiane à la liste `medianes`.

> **Rappel.**
> ```python
> numeros = []
> medianes = []
> for arr in range(1, 21):
>     ...
> ```
> Dans la boucle, le bloc est décalé de quatre espaces.

In [ ]:
numeros = []
medianes = []

for arr in range(1, 21):
    sous_table = propre.query("arrondissement == @arr")
    numeros.append(arr)
    medianes.append(sous_table["prix_m2"].median())

print(numeros)
print([round(m) for m in medianes])

#### Deux listes, c'est une de trop

Les deux listes vont ensemble : le sixième nombre de `medianes` est la médiane
du sixième numéro de `numeros`. Rien ne le garantit. Triez l'une des deux, et
tout est faux.

pandas a l'objet qu'il faut : une **Series**. C'est une colonne de valeurs où
chaque valeur porte une **étiquette**. Vous en manipulez depuis le début de
l'exercice sans lui avoir donné son nom :

| Ce que vous avez écrit | Les valeurs | Les étiquettes |
|---|---|---|
| `propre["prix_m2"]` | les prix au m² | le numéro de chaque ligne |
| `propre["type_local"].value_counts()` | les effectifs | les noms des types |
| `mois` (exercice 9) | les nombres de ventes | les numéros de mois, d'où `mois.loc[8]` |

On en fabrique une à partir de nos deux listes, les valeurs d'un côté, les
étiquettes de l'autre :

```python
prix_par_arr = pd.Series(medianes, index=numeros)
```

Les deux listes ne font plus qu'un objet, et **l'étiquette voyage avec sa
valeur** : trier, filtrer, tracer, rien ne peut plus les désaligner.

### Exercice 19 — La Series et le graphique

Construisez `prix_par_arr`, affichez-la, puis allez chercher la médiane du 19e
avec `.loc` dans `prix_19`. Enfin tracez-la en barres horizontales, triée, avec
un titre et l'unité.

> **Rappel.** Barres horizontales : `serie.sort_values().plot(kind="barh", figsize=(7, 5))`.
> Le tri croissant met le plus grand en haut. Ajoutez `plt.tight_layout()`
> avant `plt.show()` pour que rien ne soit coupé.

In [ ]:
prix_par_arr = pd.Series(medianes, index=numeros)
prix_19 = prix_par_arr.loc[19]
print(prix_par_arr.round(0))

prix_par_arr.sort_values().plot(kind="barh", figsize=(7, 5))
plt.title("Prix median au m2 par arrondissement (Paris, 2024)")
plt.xlabel("euros par m2")
plt.tight_layout()
plt.show()

In [ ]:
verifier("vingt arrondissements", len(prix_par_arr) == 20, "la boucle va de 1 a 20 : range(1, 21)")
verifier("mediane du 19e", round(prix_19) == 7836, "pd.Series(medianes, index=numeros), puis .loc[19]")

Les numéros d'arrondissement sont arrivés tout seuls en face des barres :
ce sont les étiquettes de la Series. Avec deux listes, il aurait fallu
expliquer à matplotlib laquelle va où.

Du 19e au 6e, le prix au m² **presque double**. Et le classement n'est pas
une surprise pour un Parisien : l'ouest et le centre en haut, le nord-est en
bas. Vous venez de le mesurer sur 25 000 ventes réelles.

Ce que vous avez écrit dans la boucle, découper selon une colonne, calculer
dans chaque paquet, rassembler les résultats avec leurs étiquettes, pandas
sait le faire en une seule ligne. Vous verrez cette ligne plus tard dans le
cours. Elle vous paraîtra évidente : vous venez d'en écrire le contenu à la
main.

### Exercice 20 — L'année, mois par mois

Même mécanique, sur le temps. Avant d'écrire, une prédiction : **quel mois de
2024 a compté le moins de ventes ?** Vous avez déjà croisé l'information dans
cet exercice.

In [ ]:
# Ma prediction (le mois le plus calme) :

Créez une colonne `mois` dans `propre` à partir de la date. Puis une boucle sur
`range(1, 13)` qui compte les ventes de chaque mois dans une liste. Faites-en
une Series `ventes_par_mois`, étiquetée par le numéro de mois, tracez-la en
courbe, et mettez le mois le plus calme dans `mois_calme`.

> **Rappel.** `df["date"].dt.month` ; `len()` compte les lignes d'un tableau
> filtré ; `serie.plot(kind="line", marker="o", figsize=(7, 4))`.
>
> Pour le mois le plus calme : `serie.sort_values()` trie du plus petit au plus
> grand, et `.index[0]` rend l'étiquette de la première valeur.

In [ ]:
propre["mois"] = propre["date"].dt.month

nombres = []
for m in range(1, 13):
    nombres.append(len(propre.query("mois == @m")))

ventes_par_mois = pd.Series(nombres, index=range(1, 13))
mois_calme = ventes_par_mois.sort_values().index[0]

ventes_par_mois.plot(kind="line", marker="o", figsize=(7, 4))
plt.title("Nombre de ventes d'appartements par mois (Paris, 2024)")
plt.xlabel("mois")
plt.ylabel("ventes")
plt.show()

print("mois le plus calme :", mois_calme)

In [ ]:
verifier("le mois le plus calme", mois_calme == 8, "sort_values() puis .index[0] : l'etiquette de la plus petite valeur")

Août, sans discussion : deux fois moins de ventes que les mois voisins. Le
marché immobilier part en vacances. Le prix, lui, ne bouge presque pas d'un
mois à l'autre, vous pouvez le vérifier en remplaçant `len(...)` par la
médiane de `prix_m2` dans la boucle. Un marché calme n'est pas un marché qui
baisse.

### Exercice 21 — Surface et prix

Un nuage de points croise deux grandeurs, un point par vente. Tracez le prix
en fonction de la surface. Avant cela, calculez la **corrélation** entre les
deux dans `lien`, arrondie à deux décimales, et mettez-la dans le titre avec un
f-string.

**La corrélation** est un nombre entre -1 et +1. Proche de +1 : quand l'une
monte, l'autre monte. Proche de -1 : quand l'une monte, l'autre descend. Proche
de 0 : pas de lien d'ensemble. Elle met un chiffre sur ce que l'œil croit voir
dans le nuage.

> **Rappel.** `df["a"].corr(df["b"])` ; `df.plot(kind="scatter", x="surface",
> y="prix", alpha=0.2, figsize=(7, 4))`. `alpha` rend les points translucides :
> sans lui, 25 000 points font une tache. Dans un f-string, `{lien}` est
> remplacé par la valeur.

In [ ]:
lien = round(propre["surface"].corr(propre["prix"]), 2)

propre.plot(kind="scatter", x="surface", y="prix", alpha=0.2, figsize=(7, 4))
plt.title(f"Prix selon la surface (correlation {lien})")
plt.xlabel("Surface (m2)")
plt.ylabel("Prix (euros)")
plt.show()

In [ ]:
verifier("correlation", lien == 0.88, "corr() entre surface et prix, arrondi a 2 decimales")

Une corrélation très forte, et le nuage le montre : plus c'est grand, plus
c'est cher. Ce n'est pas une découverte, mais c'est mesuré. Remarquez aussi
l'épaisseur du nuage à surface égale : pour 50 m², les prix vont du simple au
triple. C'est l'arrondissement, et c'est ce que la carte va montrer.

### Exercice 22 — La réponse au client

Le client a 400 000 €. Divisez ce budget par la Series des prix médians :
vous obtenez, pour chaque arrondissement, la surface qu'il peut acheter.
Arrondissez, tracez en barres horizontales triées, et mettez dans `meilleur`
le numéro de l'arrondissement où il achète le plus grand.

> **Rappel.** Une opération sur une Series s'applique à chaque valeur, et les
> étiquettes suivent : `400000 / prix_par_arr` rend une Series. Après
> `sort_values()`, `.index[-1]` est l'étiquette de la plus grande valeur.

In [ ]:
m2 = (400000 / prix_par_arr).round(0)
meilleur = m2.sort_values().index[-1]

m2.sort_values().plot(kind="barh", figsize=(7, 5))
plt.title("Surface achetable avec 400 000 euros, par arrondissement")
plt.xlabel("m2")
plt.tight_layout()
plt.show()

print(f"Avec 400 000 euros : {m2.loc[meilleur]:.0f} m2 dans le {meilleur}e, contre {m2.min():.0f} m2 dans le {m2.sort_values().index[0]}e")

In [ ]:
verifier("le meilleur arrondissement", meilleur == 19, "la plus grande surface est la derniere apres un tri croissant : .index[-1]")

Voilà la réponse : **51 m² dans le 19e, 27 m² dans le 6e**, pour le
même budget. Presque du simple au double, à Paris, à quelques stations de
métro d'écart.

---

## Partie 4 — La carte

Le fichier contient la position de chaque appartement vendu : `longitude` et
`latitude`. Un nuage de points avec la longitude en abscisse et la latitude
en ordonnée, c'est **une carte**. Et si on colore chaque point selon son
`prix_m2`, c'est la carte des prix de Paris.

La cellule est écrite pour vous. Le seul mot nouveau est `c=`, qui donne la
colonne qui sert à colorer les points ; `cmap` choisit la palette, et `vmin`,
`vmax` bornent l'échelle des couleurs pour que les extrêmes n'écrasent pas le
reste. Exécutez-la.

In [ ]:
propre.plot(kind="scatter", x="longitude", y="latitude",
            c="prix_m2", cmap="viridis", vmin=6000, vmax=16000,   ## la couleur = le prix au m2
            s=4, alpha=0.6, figsize=(8, 7))
plt.title("Prix au m2 des appartements vendus a Paris en 2024")
plt.xlabel("longitude")
plt.ylabel("latitude")
plt.tight_layout()
plt.show()

C'est Paris. La Seine se devine en creux, le bois de Boulogne à gauche, le
bois de Vincennes à droite. Le centre et l'ouest en jaune, le nord-est en
violet. Chaque point est une vente réelle de 2024, et c'est vous qui les avez
rendues lisibles : au départ, ce fichier ne permettait même pas de calculer un
prix moyen.

Pour finir, la même carte sur un seul arrondissement. Changez le numéro et
exécutez :

In [ ]:
arr_choisi = 11   ## changez le numero

propre.query("arrondissement == @arr_choisi").plot(
    kind="scatter", x="longitude", y="latitude",
    c="prix_m2", cmap="viridis", vmin=6000, vmax=16000,
    s=12, alpha=0.7, figsize=(7, 6))
plt.title(f"Prix au m2 dans le {arr_choisi}e arrondissement (2024)")
plt.tight_layout()
plt.show()

---

## Pour conclure

Complétez cette cellule de texte (double-clic pour l'éditer) en trois phrases,
avec vos chiffres :

- Le prix médian d'un appartement à Paris en 2024 est de … € le m². L'écart entre l'arrondissement le moins cher et le plus cher va de … à … .
- Avec 400 000 €, le client achète le plus grand dans le … arrondissement, soit environ … m².
- Une chose que ce fichier m'a apprise sur le nettoyage des données : …

## Ce que vous avez fait

Vous êtes parti d'un fichier de 61 000 lignes où le prix moyen ne se
calculait même pas. Vous avez :

- compris que chaque ligne était un lot et non une vente, et vérifié à la fin que ce n'était plus le cas ;
- corrigé sept défauts, dans l'ordre, en notant à chaque fois ce que vous retiriez et pourquoi ;
- fait la différence entre réduire son périmètre et perdre des données ;
- reconstruit à la main, avec une boucle et une Series, un calcul par groupe ;
- répondu à la question du client avec un chiffre, un graphique, et une carte.

| Vous avez utilisé | Pour |
|---|---|
| `shape`, `info()`, `head()`, `describe()` | prendre en main un fichier inconnu |
| `len()` contre `nunique()` | savoir ce qu'est une ligne |
| `duplicated()`, `drop_duplicates()` | les doublons |
| `isna().sum()`, `dropna(subset=...)` | les valeurs manquantes, après avoir compris d'où elles venaient |
| `.str.replace()`, `pd.to_numeric(errors="coerce")` | les nombres en texte, et le compte obligatoire après |
| `.str[3:5]`, `pd.to_datetime(format="mixed", dayfirst=True)`, `.dt.month` | les dates, avec un repère avant de convertir |
| `.str.strip().str.lower()` | le texte incohérent |
| `describe()`, `sort_values().head(3)`, `query()` | les valeurs aberrantes |
| `duplicated(subset=..., keep=False)` | les ventes de plusieurs lots |
| une liste, `append`, une boucle `for` | le journal, et les vingt médianes |
| `query("col == @variable")` | une variable dans un filtre |
| `pd.Series(valeurs, index=etiquettes)`, `.loc[etiquette]` | tenir ensemble des valeurs et leurs étiquettes |
| `plot(kind="hist" / "barh" / "line" / "scatter")` | quatre graphiques, quatre questions |
| `corr()` | un chiffre sur un lien |

> ⚠️ **Avant de fermer l'onglet :** vérifiez que votre notebook est bien
> enregistré dans votre Drive.

---

## Ce que vous savez faire maintenant

| Compétence | Vous voulez... | La commande |
|---|---|---|
| Charger | charger un fichier | `pd.read_csv(BASE + "ventes.csv")` |
| Explorer | voir les premières lignes | `df.head(3)` |
| Explorer | voir les dernières lignes | `df.tail(10)` |
| Explorer | connaître la taille du tableau | `df.shape` |
| Explorer | voir les colonnes et leurs types | `df.info()` |
| Explorer | lire la liste exacte des colonnes | `df.columns` |
| Explorer | compter les valeurs distinctes | `df["col"].nunique()` |
| Sélectionner | sélectionner une colonne | `df["prix"]` |
| Sélectionner | sélectionner plusieurs colonnes | `df[["qte", "prix"]]` |
| Sélectionner | sélectionner une ligne par sa position | `df.iloc[0]` |
| Sélectionner | sélectionner une case par étiquette | `df.loc[10, "prix"]` |
| Filtrer | filtrer des lignes | `df.query("prix > 50")` |
| Filtrer | combiner deux conditions | `df.query("qte >= 100 and prix > 2")` |
| Filtrer | filtrer sur une liste de valeurs | `df.query("pays in ['France', 'Belgique']")` |
| Filtrer | filtrer sur un intervalle | `df.query("50 <= qte <= 100")` |
| Filtrer | obtenir une réponse Vrai/Faux par ligne | `df["prix"] > 50` |
| Filtrer | compter les lignes qui remplissent une condition | `(df["prix"] > 50).sum()` |
| Résumer | obtenir le résumé chiffré | `df[["qte", "prix"]].describe()` |
| Résumer | compter les catégories | `df["pays"].value_counts()` |
| Résumer | calculer un indicateur | `df["prix"].mean()`, `.median()`, `.max()`, `.sum()` |
| Calculer | créer une colonne calculée | `df["ca"] = df["qte"] * df["prix"]` |
| Nettoyer | repérer les valeurs manquantes | `df.isna().sum()` |
| Nettoyer | supprimer les lignes incomplètes | `df.dropna(subset=["client_id"])` |
| Nettoyer | remplacer les valeurs manquantes | `df["prix"].fillna(0)` |
| Nettoyer | compter les doublons | `df.duplicated().sum()` |
| Nettoyer | supprimer les doublons | `df.drop_duplicates()` |
| Nettoyer | convertir du texte en nombre | `pd.to_numeric(col, errors="coerce")` |
| Nettoyer | convertir du texte en date | `pd.to_datetime(col, format="mixed", dayfirst=True)` |
| Nettoyer | extraire le mois d'une date | `df["date"].dt.month` |
| Nettoyer | nettoyer du texte | `col.str.strip().str.lower()` |
| Nettoyer | enlever un morceau de texte | `col.str.replace(" EUR", "")` |
| Enrichir | classer selon une condition | `np.where(cond, "oui", "non")` |

Deux commandes vont plus loin et vous attendent dans la **feuille facultative** :
`np.select` pour classer selon plusieurs conditions, et `pd.cut` pour découper
une colonne de nombres en tranches.

## Les réflexes à emporter

Devant un fichier inconnu, toujours dans cet ordre : **`shape`, `info()`,
`head(3)`, `describe()`**. Trente secondes, et vous savez de quoi vous parlez.
Et la première question à se poser : **une ligne, c'est quoi exactement ?**

**Le nettoyage est un pipeline, pas une série de bricolages.** Écrivez-le dans
l'ordre, de haut en bas, en repartant toujours du fichier brut. Le jour où on
vous livre le fichier du mois suivant, vous relancez le notebook et c'est fini.

Et **notez toujours combien de lignes vous perdez à chaque étape**. Un
nettoyage qui fait disparaître 40 % des données n'est pas un nettoyage, c'est
une erreur.